## Import Libraries

### Import all necessary libraries for data cleaning and preprocessing

In [1]:
!pip install pandas numpy geopy scikit-learn


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

## Data Cleaning and Encoding

### Clean data by dropping duplicated, missing values, 

## Clean and Convert Data

In [3]:
def count_items(x):
            if pd.isna(x):
                return 0
            return len(str(x).split(';'))

In [4]:
for district_num in range(1, 29):  
    try:  
        df = pd.read_csv(f"../datasets/updated_coordinates/district{district_num}.csv")

        # Remove commas, dollar signs, etc., and convert to numeric
        for col in ["Transacted Price ($)", "Area (SQFT)", "Unit Price ($ PSF)", "Area (SQM)", "Unit Price ($ PSM)"]:
            df[col] = df[col].replace(r'[\$,]', '', regex=True).astype(float)
            df = df.dropna(subset=["Unit Price ($ PSF)", "Area (SQFT)"])

        df = df[~df['Floor Level'].isin(['B1', 'B2', 'G', 'PH'])]
        df['Sale Date'] = pd.to_datetime(df['Sale Date'], format='%b-%y', errors='coerce')
        df['Sale Months Since Sep 2025'] = round((pd.Timestamp("2025-09-01") - df['Sale Date'] ) / pd.Timedelta(days=30))
    

        df['Num MRT Within 1km'] = df['Nearest MRT Stations'].apply(count_items)
        df['Num Hawker Within 1km'] = df['Nearby Hawker Centers'].apply(count_items)
        df['Num Malls Within 1km'] = df['Shopping Malls Within Radius of 1km'].apply(count_items)
        df['Num Hospitals Within 5km'] = df['Hospitals Within Radius of 5km'].apply(count_items)
        df['Num Schools Within 2km'] = df['Schools Within Radius of 2km'].apply(count_items)
        df['Num Parks Within 1km'] = df['Parks Within Radius of 1km'].apply(count_items)

        df.drop(columns=['Nett Price($)', 'Street Name', 'Nearest MRT Stations', 'Nearby Hawker Centers', 'Shopping Malls Within Radius of 1km', 'Hospitals Within Radius of 5km', 'Schools Within Radius of 2km', 'Parks Within Radius of 1km', 'Number of Units'], inplace=True)

        df.to_csv(f"../datasets/normalized_data/district{district_num}_normalized.csv", index=False)

    except Exception as e:
        print(f"Error processing district {district_num}: {str(e)}")
        continue

Error processing district 24: [Errno 2] No such file or directory: '../datasets/updated_coordinates/district24.csv'


## Encode

In [5]:
def floor_level_to_num(floor_str):
    """
    Convert floor level string to numerical category
    B1-B5: 0 (Basement)
    01-05: 1
    06-10: 2
    11-15: 3
    16-20: 4
    21-25: 5
    26-30: 6
    31-35: 7
    36-40: 8
    41-45: 9
    46-50: 10
    etc.
    """
    if pd.isna(floor_str):
        return np.nan
    
    floor_str = str(floor_str).strip().upper()
    
    # Handle basement floors
    if 'B' in floor_str:
        return 0
    
    # Extract the starting floor number
    try:
        # Handle formats like "01 to 05", "01-05", etc.
        start_floor = int(floor_str.split()[0].split('-')[0])
        
        # Calculate category: floors 1-5 = 1, 6-10 = 2, etc.
        category = ((start_floor - 1) // 5) + 1
        return category
    except:
        return np.nan

def encode_type_of_sale(value):
    """
    Convert Type of Sale to numerical
    New Sale: 0
    Resale: 1
    """
    if pd.isna(value):
        return np.nan
    value_lower = str(value).lower()
    if 'new' in value_lower:
        return 0
    elif 'resale' in value_lower:
        return 1
    else:
        return np.nan

def encode_property_type(value):
    """
    Convert Property Type to numerical
    Apartment: 0
    Condominium: 1
    """
    if pd.isna(value):
        return np.nan
    value_lower = str(value).lower()
    if 'apartment' in value_lower:
        return 0
    elif 'condominium' in value_lower:
        return 1
    else:
        return np.nan

def encode_market_segment(value):
    """
    Convert Market Segment to numerical
    This depends on your specific categories
    You may need to adjust based on actual values
    """
    if pd.isna(value):
        return np.nan
    
    value_lower = str(value).lower()
    
    # Define your mapping based on actual categories in your data
    segment_mapping = {
        'ccr': 0,  # Core Central Region
        'core central region': 0,
        'rcr': 1,  # Rest of Central Region
        'rest of central region': 1,
        'ocr': 2,  # Outside Central Region
        'outside central region': 2
    }
    
    for key, num in segment_mapping.items():
        if key in value_lower:
            return num
    
    return np.nan

In [6]:
# Main processing loop
for district_num in range(1, 29):
    try:
        print(f"Processing district {district_num}...")
        cleaned_df = pd.read_csv(f"../datasets/normalized_data/district{district_num}_normalized.csv")
        
        # Create a copy to work with
        encoded_df = cleaned_df.copy()
        if 'Floor Level' in encoded_df.columns:
            encoded_df['Floor_Level_Category'] = encoded_df['Floor Level'].apply(floor_level_to_num)
        if 'Type of Sale' in encoded_df.columns:
            encoded_df['Type_of_Sale_Encoded'] = encoded_df['Type of Sale'].apply(encode_type_of_sale)
            # Drop one-hot encoded columns if they exist
            type_sale_cols = [col for col in encoded_df.columns if col.startswith('Type of Sale_')]
            encoded_df = encoded_df.drop(type_sale_cols, axis=1)
        if 'Property Type' in encoded_df.columns:
            encoded_df['Property_Type_Encoded'] = encoded_df['Property Type'].apply(encode_property_type)
            # Drop one-hot encoded columns if they exist
            property_type_cols = [col for col in encoded_df.columns if col.startswith('Property Type_')]
            encoded_df = encoded_df.drop(property_type_cols, axis=1)
        if 'Market Segment' in encoded_df.columns:
            encoded_df['Market_Segment_Encoded'] = encoded_df['Market Segment'].apply(encode_market_segment)
            # Drop one-hot encoded columns if they exist
            market_segment_cols = [col for col in encoded_df.columns if col.startswith('Market Segment_')]
            encoded_df = encoded_df.drop(market_segment_cols, axis=1)
        if 'Type of Area' in encoded_df.columns:
            # Check if it needs encoding
            unique_values = encoded_df['Type of Area'].unique()
            print(f"  Type of Area unique values: {unique_values}")
            
            # If it's categorical like "Strata", "Land", etc., encode it
            area_type_mapping = {
                'strata': 0,
                'land': 1,
                # Add more as needed
            }
            encoded_df['Type_of_Area_Encoded'] = encoded_df['Type of Area'].apply(
                lambda x: area_type_mapping.get(str(x).lower(), np.nan) if pd.notna(x) else np.nan
            )
        
        encoded_df.to_csv(f"../datasets/normalized_data/district{district_num}_normalized.csv", index=False)
        
        print(f"✓ Successfully processed district {district_num}")
        print(f"  Original columns: {len(cleaned_df.columns)}")
        print(f"  New columns: {len(encoded_df.columns)}")
        print(f"  Added encodings: {len(encoded_df.columns) - len(cleaned_df.columns)}")
        print()
    
    except FileNotFoundError:
        print(f"⚠ File not found for district {district_num}")
        continue
    except Exception as e:
        print(f"✗ Error processing district {district_num}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue

print("="*80)
print("ENCODING COMPLETE!")
print("="*80)

Processing district 1...
  Type of Area unique values: ['Strata']
✓ Successfully processed district 1
  Original columns: 25
  New columns: 30
  Added encodings: 5

Processing district 2...
  Type of Area unique values: ['Strata']
✓ Successfully processed district 2
  Original columns: 25
  New columns: 30
  Added encodings: 5

Processing district 3...
  Type of Area unique values: ['Strata']
✓ Successfully processed district 3
  Original columns: 25
  New columns: 30
  Added encodings: 5

Processing district 4...
  Type of Area unique values: ['Strata']
✓ Successfully processed district 4
  Original columns: 25
  New columns: 30
  Added encodings: 5

Processing district 5...
  Type of Area unique values: ['Strata']
✓ Successfully processed district 5
  Original columns: 25
  New columns: 30
  Added encodings: 5

Processing district 6...
  Type of Area unique values: ['Strata']
✓ Successfully processed district 6
  Original columns: 25
  New columns: 30
  Added encodings: 5

Processing

In [7]:
for district_num in range(1, 29):
    try:
        encoded_df = pd.read_csv(f"../datasets/cleaned_data/district{district_num}_cleaned.csv")
        
        # List of columns to drop if they exist
        columns_to_drop = [
            'Postal District',
            'Property Type',
            'Tenure',
            'Market Segment',
            'Floor Level',
            'Type of Sale',
            'Type of Area',
            'Postal_District_Encoded',
            'Type_of_Area_Encoded'
        ]
        
        # Find which columns actually exist in the dataframe
        existing_columns_to_drop = [col for col in columns_to_drop if col in encoded_df.columns]
        
        # Drop the existing columns
        if existing_columns_to_drop:
            encoded_df.drop(columns=existing_columns_to_drop, inplace=True)
            print(f"District {district_num}: Dropped columns {existing_columns_to_drop}")
        else:
            print(f"District {district_num}: No columns to drop")
        
        # Save the modified dataframe
        encoded_df.to_csv(f"../datasets/cleaned_data/district{district_num}_cleaned.csv", index=False)
        
    except FileNotFoundError:
        print(f"File not found for district {district_num}")
        continue
    except Exception as e:
        print(f"Error processing district {district_num}: {str(e)}")
        continue

print("Column dropping complete!")

District 1: No columns to drop
District 2: No columns to drop
District 3: No columns to drop
District 4: No columns to drop
District 5: No columns to drop
District 6: No columns to drop
District 7: No columns to drop
District 8: No columns to drop
District 9: No columns to drop
District 10: No columns to drop
District 11: No columns to drop
District 12: No columns to drop
District 13: No columns to drop
District 14: No columns to drop
District 15: No columns to drop
District 16: No columns to drop
District 17: No columns to drop
District 18: No columns to drop
District 19: No columns to drop
District 20: No columns to drop
District 21: No columns to drop
District 22: No columns to drop
District 23: No columns to drop
File not found for district 24
District 25: No columns to drop
District 26: No columns to drop
District 27: No columns to drop
District 28: No columns to drop
Column dropping complete!


In [8]:
encoded_df.head()

,Project Name,Transacted Price ($),Area (SQFT),Unit Price ($ PSF),Sale Date,Area (SQM),Unit Price ($ PSM),Latitude,Longitude,Full Address,...,Num MRT Within 1km,Num Hawker Within 1km,Num Malls Within 1km,Num Hospitals Within 5km,Num Schools Within 2km,Num Parks Within 1km,Floor_Level_Category,Type_of_Sale_Encoded,Property_Type_Encoded,Market_Segment_Encoded
0,THE WARREN,1220000.0,1044.11,1168.0,2025-09-01,97.0,12577.0,1.385846,103.742424,39 CHOA CHU KANG LOOP THE WARREN SINGAPORE 689676,...,3,0,4,1,16,1,2.0,1.0,1,2
1,MERALODGE,2358000.0,1636.13,1441.0,2025-09-01,152.0,15513.0,1.354164,103.761454,83 HILLVIEW AVENUE MERALODGE SINGAPORE 669583,...,0,0,2,3,11,8,1.0,1.0,1,2
2,THE SKYWOODS,1680000.0,1011.82,1660.0,2025-09-01,94.0,17872.0,1.365729,103.771413,9 DAIRY FARM HEIGHTS THE SKYWOODS SINGAPORE 67...,...,2,0,2,3,6,10,1.0,1.0,1,2
3,THE MYST,1595000.0,678.13,2352.0,2025-09-01,63.0,25317.0,1.373499,103.764142,800 UPPER BUKIT TIMAH ROAD THE MYST (U/C) SING...,...,6,0,3,3,16,8,5.0,0.0,1,2
4,THE TENNERY,825000.0,613.55,1345.0,2025-09-01,57.0,14474.0,1.379727,103.760191,3 WOODLANDS ROAD THE TENNERY SINGAPORE 677901,...,7,1,5,1,18,2,3.0,1.0,0,2


In [9]:
encoded_df.columns.tolist()

['Project Name',
 'Transacted Price ($)',
 'Area (SQFT)',
 'Unit Price ($ PSF)',
 'Sale Date',
 'Area (SQM)',
 'Unit Price ($ PSM)',
 'Latitude',
 'Longitude',
 'Full Address',
 'Dist to CBD in Km',
 'Sale Months Since Sep 2025',
 'Num MRT Within 1km',
 'Num Hawker Within 1km',
 'Num Malls Within 1km',
 'Num Hospitals Within 5km',
 'Num Schools Within 2km',
 'Num Parks Within 1km',
 'Floor_Level_Category',
 'Type_of_Sale_Encoded',
 'Property_Type_Encoded',
 'Market_Segment_Encoded']

## Z-Score Normalization

In [10]:
for district_num in range(1, 29):
    try:
        normalized_df = pd.read_csv(f"../datasets/normalized_data/district{district_num}_normalized.csv")

        num_cols = ['Transacted Price ($)',
                    'Area (SQFT)',
                    'Unit Price ($ PSF)',
                    'Dist to CBD in Km',
                    'Sale Months Since Sep 2025',
                    'Num MRT Within 1km',
                    'Num Hawker Within 1km',
                    'Num Malls Within 1km',
                    'Num Hospitals Within 5km',
                    'Num Schools Within 2km',
                    'Num Parks Within 1km',
                    'Floor_Level_Category',
                    'Type_of_Sale_Encoded',
                    'Property_Type_Encoded',
                    'Market_Segment_Encoded']

        scaler = StandardScaler()
        normalized_df[num_cols] = scaler.fit_transform(normalized_df[num_cols])
        normalized_df = normalized_df[num_cols]

        normalized_df.to_csv(f"../datasets/normalized_data/district{district_num}_normalized.csv")
    
    except Exception as e:
        print(f"Error processing district {district_num}: {str(e)}")
        continue

Error processing district 24: [Errno 2] No such file or directory: '../datasets/normalized_data/district24_normalized.csv'


In [11]:
normalized_df.head()

,Transacted Price ($),Area (SQFT),Unit Price ($ PSF),Dist to CBD in Km,Sale Months Since Sep 2025,Num MRT Within 1km,Num Hawker Within 1km,Num Malls Within 1km,Num Hospitals Within 5km,Num Schools Within 2km,Num Parks Within 1km,Floor_Level_Category,Type_of_Sale_Encoded,Property_Type_Encoded,Market_Segment_Encoded
0,2.809214,2.070833,0.237097,-2.772179,-1.711465,-1.632820,-1.555130,-0.328072,-0.506403,-0.209238,2.226429,-0.905237,0.03705,0.739800,0.0
1,1.766767,0.629820,1.472274,0.512166,-1.711465,0.340576,-0.120714,-0.328072,-0.506403,-0.506668,-0.355651,-0.166575,0.03705,-1.351717,0.0
2,0.550579,1.221664,-1.379469,-1.124401,-1.711465,-1.632820,-1.555130,-1.985871,2.556176,-1.696390,1.710013,-0.905237,0.03705,0.739800,0.0
3,1.136956,0.037953,2.014019,1.177032,-1.711465,0.340576,-0.120714,-0.328072,1.024887,-0.804098,-0.355651,0.572086,0.03705,0.739800,0.0
4,0.116227,-0.425228,1.329254,1.177032,-1.711465,0.340576,-0.120714,-0.328072,1.024887,-0.804098,-0.355651,1.310748,0.03705,0.739800,0.0


In [12]:
normalized_df.columns.tolist()

['Transacted Price ($)',
 'Area (SQFT)',
 'Unit Price ($ PSF)',
 'Dist to CBD in Km',
 'Sale Months Since Sep 2025',
 'Num MRT Within 1km',
 'Num Hawker Within 1km',
 'Num Malls Within 1km',
 'Num Hospitals Within 5km',
 'Num Schools Within 2km',
 'Num Parks Within 1km',
 'Floor_Level_Category',
 'Type_of_Sale_Encoded',
 'Property_Type_Encoded',
 'Market_Segment_Encoded']

In [13]:
print("="*80)
print("LOADING NORMALIZED DATA FROM ALL DISTRICTS")
print("="*80)

districts = []
for i in range(1, 29):
    try: 
        df = pd.read_csv(f'../datasets/normalized_data/district{i}_normalized.csv')
        df['District'] = i
        districts.append(df)
    except FileNotFoundError:
        print(f"District {i} file not found")

combined_df = pd.concat(districts, ignore_index=True)  
combined_df = combined_df.dropna()
combined_df.drop(['Unnamed: 0'], axis=1, inplace=True, errors='ignore')
combined_df.to_csv(f"../datasets/normalized_data/combined_normalized.csv", index=False)
combined_df

LOADING NORMALIZED DATA FROM ALL DISTRICTS
District 24 file not found


,Transacted Price ($),Area (SQFT),Unit Price ($ PSF),Dist to CBD in Km,Sale Months Since Sep 2025,Num MRT Within 1km,Num Hawker Within 1km,Num Malls Within 1km,Num Hospitals Within 5km,Num Schools Within 2km,Num Parks Within 1km,Floor_Level_Category,Type_of_Sale_Encoded,Property_Type_Encoded,Market_Segment_Encoded,District
0,-0.211237,-0.460443,0.774350,1.347673,-1.117103,-1.177227,-1.175060,-1.296450,-1.143547,-1.118512,-1.374335,-1.619747,-1.228179,-0.334688,1.124971,1
1,-0.087630,-0.460443,1.240198,1.347673,-1.117103,-1.177227,-1.175060,-1.296450,-1.143547,-1.118512,-1.374335,0.947931,-1.228179,-0.334688,1.124971,1
2,-0.303362,-0.331436,-0.058154,-0.618836,-1.117103,-0.723616,-0.407922,0.640686,0.276567,-0.286674,0.161507,-0.152502,0.814214,2.987854,-0.888912,1
3,-0.100129,-0.442004,1.103809,1.347673,-1.117103,-1.177227,-1.175060,-1.296450,-1.143547,-1.118512,-1.374335,0.214309,-1.228179,-0.334688,1.124971,1
4,1.493910,2.267081,-0.950882,-0.363508,-1.117103,0.183604,-0.407922,0.156402,-0.670176,-0.286674,-0.030473,-1.619747,0.814214,-0.334688,-0.888912,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108728,-0.947937,-0.579614,-0.985079,0.574579,1.673034,0.833925,1.313703,-0.328072,-0.506403,0.980484,-0.613859,1.310748,0.037050,-1.351717,0.000000,28
108729,-0.937078,0.166615,-2.397949,-1.124401,1.673034,-1.632820,-1.555130,-1.985871,2.556176,-1.696390,1.710013,-0.905237,0.037050,0.739800,0.000000,28
108730,-1.023949,-0.888409,-0.204967,0.512166,1.673034,0.340576,-0.120714,-0.328072,-0.506403,-0.506668,-0.355651,2.049409,0.037050,-1.351717,0.000000,28
108731,0.702603,1.504712,-1.513821,0.246256,1.673034,0.833925,1.313703,1.329727,-0.506403,0.980484,-0.872068,-0.905237,0.037050,0.739800,0.000000,28
